# Notebook 2 — DICOM Preprocessing & Cache
**RSNA Intracranial Hemorrhage Detection**

This notebook converts raw DICOM files into 3-channel windowed **NPY arrays**
(float32, 0–1 range, shape 256×256×3) and saves them as a Kaggle output artifact.

**Subsequent notebooks add this notebook's output as an input dataset.**

### Critical workflow
1. **Pilot first**: Set `SUBSET_FRAC = 0.1` to validate pipeline on ~75K images
2. Once pilot passes health check, set `SUBSET_FRAC = 1.0` and re-run
3. Run via **Save Version → Save & Run All** (background commit)
4. In the next notebook, click **Add Data → Notebook Output Files** and add this run

### Key design decisions
- **NPY format**: Lossless float32 arrays — no PNG quantization, faster I/O,
  exact value preservation for reproducible training.
- **Patient-level split**: `GroupShuffleSplit` on PatientID from DICOM headers.
  Most patients have only 1–2 slices (mean ≈ 1.2), so leakage risk is modest
  (~0.01–0.02 AUC), but group-split is used for methodological rigor.
- **Normalization stats**: Dataset-specific mean/std computed on cached arrays.
- **Checkpointing**: Progress saved every 10K images — session crash resumes
  from last checkpoint, not from zero.
- **Robust metadata**: WindowCenter/Width handled as scalar or list (DICOM quirk).

In [ ]:
# ── 0. Config ──────────────────────────────────────────────────────────────
import os, gc, glob, random
from pathlib import Path

# ─── Kaggle paths ────────────────────────────────────────────────────────
BASE      = Path('/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection')
TRAIN_CSV = BASE / 'stage_2_train.csv'
TRAIN_DIR = BASE / 'stage_2_train/'

# ─── Output paths ────────────────────────────────────────────────────────
OUT_DIR   = Path('/kaggle/working/cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Processing config ───────────────────────────────────────────────────
IMG_SIZE     = 256        # resize target (pixels)
SUBSET_FRAC  = 0.1        # ⚠️ START WITH 0.1 (pilot ~75K). Set 1.0 for full run.
VAL_FRAC     = 0.15       # 15% validation
SEED         = 42
NUM_WORKERS  = 4          # parallel DICOM workers
CHECKPOINT_EVERY = 10000  # save progress index every N images

# CT windows: (window_center, window_width)
WINDOWS = [
    (40,   80),    # brain
    (75,  215),    # subdural
    (40,  380),    # soft tissue
]

print(f'Config OK. Output dir: {OUT_DIR}')
print(f'Mode: {"PILOT (10%)" if SUBSET_FRAC < 1.0 else "FULL DATASET"}')

In [ ]:
# ── 1. Load and prepare label dataframe ──────────────────────────────────
import numpy as np
import pandas as pd

raw = pd.read_csv(TRAIN_CSV)
raw[['image_id', 'subtype']] = raw['ID'].str.rsplit('_', n=1, expand=True)

# Drop duplicate (image_id, subtype) rows present in the RSNA CSV
raw = raw.drop_duplicates(subset=['image_id', 'subtype'], keep='first')

SUBTYPES = ['any', 'epidural', 'intraparenchymal',
            'intraventricular', 'subarachnoid', 'subdural']

df = raw.pivot(index='image_id', columns='subtype', values='Label').reset_index()
df.columns.name = None
for col in SUBTYPES:
    df[col] = df[col].astype(int)

# Subsample if requested — stratified on 'any' to preserve class balance
if SUBSET_FRAC < 1.0:
    from sklearn.model_selection import train_test_split
    df, _ = train_test_split(df, train_size=SUBSET_FRAC,
                             stratify=df['any'], random_state=SEED)
    df = df.reset_index(drop=True)
    print(f'Using {len(df):,} images ({SUBSET_FRAC*100:.0f}% stratified subset)')
else:
    print(f'Using full dataset: {len(df):,} images')

print(f'Positive rate: {df["any"].mean()*100:.2f}%')

In [ ]:
# ── 2. Extract PatientID from DICOM headers ──────────────────────────────
# We need PatientID for patient-level train/val split (GroupShuffleSplit).
# Note: Most patients in this dataset have only 1-2 slices (mean ≈ 1.2),
# so leakage risk is modest, but we still group-split for rigor.
import pydicom
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm


def _extract_patient_id(image_id):
    """Read DICOM header (no pixels) to get PatientID. Fast (~1ms each)."""
    dcm_path = str(TRAIN_DIR / f'{image_id}.dcm')
    try:
        dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        pid = str(getattr(dcm, 'PatientID', 'UNKNOWN'))
        return image_id, pid
    except Exception:
        return image_id, 'UNKNOWN'


all_ids = df['image_id'].tolist()
patient_map = {}

print(f'Extracting PatientID from {len(all_ids):,} DICOM headers...')
print(f'(metadata-only read — no pixel data loaded, ~10-20 min for full dataset)')

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(_extract_patient_id, img_id): img_id
               for img_id in all_ids}
    with tqdm(total=len(all_ids), unit='dcm', desc='PatientID scan') as pbar:
        for future in as_completed(futures):
            img_id, pid = future.result()
            patient_map[img_id] = pid
            pbar.update(1)

df['patient_id'] = df['image_id'].map(patient_map)
n_patients = df['patient_id'].nunique()
n_unknown  = (df['patient_id'] == 'UNKNOWN').sum()

print(f'\nUnique patients  : {n_patients:,}')
print(f'Slices / patient : {len(df) / n_patients:.1f} (mean)')
print(f'Unknown PIDs     : {n_unknown} (will be assigned to training set)')

if n_unknown > 0:
    print(f'  WARNING: {n_unknown} images had no PatientID in DICOM header.')

In [ ]:
# ── 3. Patient-level train/val split (GroupShuffleSplit) ──────────────────
# GroupShuffleSplit ensures ALL slices from a given patient stay in the same fold.
# Note: In this dataset most patients have only 1-2 slices (mean ≈ 1.2),
# so practical leakage from a naive split would be modest (~0.01-0.02 AUC).
# We still use patient-level splitting for methodological correctness.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
groups = df['patient_id'].values

train_idx, val_idx = next(gss.split(df, y=df['any'], groups=groups))

train_df = df.iloc[train_idx].copy(); train_df['split'] = 'train'
val_df   = df.iloc[val_idx].copy();   val_df['split']   = 'val'
df_split = pd.concat([train_df, val_df], ignore_index=True)

# Verify no patient leakage
train_p = set(train_df['patient_id'])
val_p   = set(val_df['patient_id'])
leaked  = train_p & val_p
assert len(leaked) == 0, f'PATIENT LEAKAGE: {len(leaked)} patients in both splits!'

print(f'Train: {len(train_df):,} images  |  Val: {len(val_df):,} images')
print(f'Train patients: {len(train_p):,}  |  Val patients: {len(val_p):,}')
print(f'Patient leakage : {len(leaked)} (MUST be 0) ✓')

In [ ]:
# ── 4. Core DICOM → NPY conversion function ──────────────────────────────
import cv2
import pydicom


def _to_scalar(val):
    """Safely extract scalar from DICOM field (may be list or MultiValue)."""
    if isinstance(val, (list, pydicom.multival.MultiValue)):
        return float(val[0])
    return float(val)


def apply_window(img_hu: np.ndarray, wc: float, ww: float) -> np.ndarray:
    lo = wc - ww / 2
    hi = wc + ww / 2
    return np.clip((img_hu - lo) / (hi - lo), 0.0, 1.0)


def dicom_to_npy(image_id: str,
                 dcm_dir: Path,
                 out_dir: Path,
                 size: int = 256) -> bool:
    """
    Read one DICOM, apply 3 CT windows, stack as (H, W, 3) float32 [0,1],
    resize, save as .npy. Returns True on success.
    """
    out_path = out_dir / f'{image_id}.npy'
    if out_path.exists():          # skip already-processed files (resume-safe)
        return True

    dcm_path = dcm_dir / f'{image_id}.dcm'
    if not dcm_path.exists():
        return False

    try:
        dcm = pydicom.dcmread(str(dcm_path))
        img = dcm.pixel_array.astype(np.float32)

        # Robust rescale — handle scalar or list DICOM fields
        slope = _to_scalar(getattr(dcm, 'RescaleSlope', 1))
        inter = _to_scalar(getattr(dcm, 'RescaleIntercept', 0))
        img   = img * slope + inter          # → Hounsfield Units

        channels = []
        for wc, ww in WINDOWS:
            ch = apply_window(img, wc, ww)
            ch = cv2.resize(ch, (size, size), interpolation=cv2.INTER_AREA)
            channels.append(ch)

        img_3ch = np.stack(channels, axis=-1).astype(np.float32)  # (H,W,3) in [0,1]
        np.save(str(out_path), img_3ch)
        return True
    except Exception as e:
        print(f'  ERROR {image_id}: {e}')
        return False


print('Conversion function defined (NPY float32, lossless).')

In [ ]:
# ── 5. Parallel conversion with checkpointing ────────────────────────────
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import json as _json_ckpt

PROGRESS_FILE = Path('/kaggle/working/cache_progress.json')

def _worker(image_id):
    """Top-level function required for ProcessPoolExecutor pickling."""
    return image_id, dicom_to_npy(image_id, TRAIN_DIR, OUT_DIR, IMG_SIZE)


all_ids = df_split['image_id'].tolist()

# Resume: skip IDs that already have cached .npy files
already_done = {p.stem for p in OUT_DIR.glob('*.npy')}
remaining_ids = [img_id for img_id in all_ids if img_id not in already_done]

print(f'Total: {len(all_ids):,}  |  Already cached: {len(already_done):,}  |  Remaining: {len(remaining_ids):,}')
print(f'Workers: {NUM_WORKERS}  |  Output size: {IMG_SIZE}x{IMG_SIZE}  |  Format: NPY float32')
print(f'Checkpoint every: {CHECKPOINT_EVERY:,} images')

failed_ids = []
processed  = len(already_done)

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(_worker, img_id): img_id for img_id in remaining_ids}
    with tqdm(total=len(remaining_ids), unit='img', initial=0) as pbar:
        for future in as_completed(futures):
            img_id, ok = future.result()
            if not ok:
                failed_ids.append(img_id)
            processed += 1
            pbar.update(1)

            # Periodic checkpoint
            if processed % CHECKPOINT_EVERY == 0:
                ckpt = {'processed': processed, 'failed': len(failed_ids)}
                with open(PROGRESS_FILE, 'w') as f:
                    _json_ckpt.dump(ckpt, f)

# Final checkpoint
ckpt = {'processed': processed, 'failed': len(failed_ids), 'done': True}
with open(PROGRESS_FILE, 'w') as f:
    _json_ckpt.dump(ckpt, f)

print(f'\nDone. Processed: {processed:,}  |  Failed: {len(failed_ids)}')
if failed_ids:
    print('  Failed IDs (first 10):', failed_ids[:10])

gc.collect()

In [ ]:
# ── 6. Build final manifest CSV ───────────────────────────────────────────
# The manifest maps image_id → cached NPY path + label columns + split + patient_id

df_split['npy_path'] = df_split['image_id'].apply(
    lambda x: str(OUT_DIR / f'{x}.npy')
)

# Drop rows where NPY wasn't created (e.g. missing/corrupted DICOM)
df_split['npy_exists'] = df_split['npy_path'].apply(os.path.exists)
missing = (~df_split['npy_exists']).sum()
if missing:
    print(f'Dropping {missing} rows with missing NPY file')
df_split = df_split[df_split['npy_exists']].drop(columns='npy_exists').reset_index(drop=True)

# Verify patient-level integrity after dropping
train_pids_final = set(df_split[df_split['split'] == 'train']['patient_id'])
val_pids_final   = set(df_split[df_split['split'] == 'val']['patient_id'])
assert len(train_pids_final & val_pids_final) == 0, 'Patient leakage after dropping missing!'

# Save manifest (now includes patient_id column)
manifest_path = Path('/kaggle/working/manifest.csv')
df_split.to_csv(manifest_path, index=False)

print(f'Manifest saved: {manifest_path}')
print(f'Rows: {len(df_split):,}')
print(f'Columns: {list(df_split.columns)}')
df_split.head()

In [ ]:
# ── 7. Post-cache verification ─────────────────────────────────────────────
# Verify random cached files: shape, dtype, value range, channel integrity.
import matplotlib.pyplot as plt

N_VERIFY = 20
verify_sample = df_split.sample(min(N_VERIFY, len(df_split)), random_state=SEED)

shapes_ok, dtype_ok, range_ok = 0, 0, 0
bad_files = []

for _, row in verify_sample.iterrows():
    arr = np.load(row['npy_path'])
    s_ok = (arr.shape == (IMG_SIZE, IMG_SIZE, 3))
    d_ok = (arr.dtype == np.float32)
    r_ok = (arr.min() >= 0.0 and arr.max() <= 1.0)
    shapes_ok += s_ok
    dtype_ok  += d_ok
    range_ok  += r_ok
    if not (s_ok and d_ok and r_ok):
        bad_files.append((row['image_id'], arr.shape, arr.dtype, arr.min(), arr.max()))

print(f'Verified {len(verify_sample)} random cached files:')
print(f'  Shape  ({IMG_SIZE},{IMG_SIZE},3) : {shapes_ok}/{len(verify_sample)}')
print(f'  Dtype  float32          : {dtype_ok}/{len(verify_sample)}')
print(f'  Range  [0, 1]           : {range_ok}/{len(verify_sample)}')
if bad_files:
    print(f'\n  ⚠️ Bad files:')
    for bf in bad_files:
        print(f'    {bf}')
else:
    print('  ✅ All files valid')

# Visual sanity check — show 3 positive samples
vis_samples = df_split[df_split['any'] == 1].sample(3, random_state=SEED)
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
for ax, (_, row) in zip(axes, vis_samples.iterrows()):
    img = np.load(row['npy_path'])  # (H,W,3) float32 [0,1]
    ax.imshow(img)
    subtypes_present = [s for s in SUBTYPES if row[s]]
    ax.set_title('  '.join(subtypes_present), fontsize=8)
    ax.axis('off')
plt.suptitle('Cached NPY sanity check (positive samples)', fontsize=11)
plt.tight_layout()
plt.savefig('/kaggle/working/cache_sanity_check.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. Compute dataset-specific normalization stats ───────────────────────
# Compute actual mean/std on our 3-channel windowed NPY arrays (already [0,1]).
# These will be saved and loaded by downstream notebooks.
import json as _json

NORM_SAMPLE_SIZE = 5000
norm_sample = df_split.sample(min(NORM_SAMPLE_SIZE, len(df_split)),
                               random_state=SEED)

channel_sums    = np.zeros(3, dtype=np.float64)
channel_sq_sums = np.zeros(3, dtype=np.float64)
n_pixels = 0

print(f'Computing normalization stats on {len(norm_sample):,} images...')
for _, row in tqdm(norm_sample.iterrows(), total=len(norm_sample),
                    desc='Norm stats', unit='img'):
    img = np.load(row['npy_path']).astype(np.float64)  # (H,W,3) already [0,1]
    channel_sums    += img.sum(axis=(0, 1))
    channel_sq_sums += (img ** 2).sum(axis=(0, 1))
    n_pixels += img.shape[0] * img.shape[1]

dataset_mean = (channel_sums / n_pixels).tolist()
dataset_std  = (np.sqrt(channel_sq_sums / n_pixels - (channel_sums / n_pixels) ** 2)).tolist()

norm_stats = {
    'mean':        [round(m, 6) for m in dataset_mean],
    'std':         [round(s, 6) for s in dataset_std],
    'n_images':    len(norm_sample),
    'n_pixels':    int(n_pixels),
    'imagenet_mean': [0.485, 0.456, 0.406],
    'imagenet_std':  [0.229, 0.224, 0.225],
    'note': 'Use dataset-specific stats for best results. '
            'ImageNet stats are provided as fallback for comparison.',
}

norm_path = Path('/kaggle/working/normalization_stats.json')
with open(norm_path, 'w') as f:
    _json.dump(norm_stats, f, indent=2)

print(f'\nDataset mean: {norm_stats["mean"]}')
print(f'Dataset std : {norm_stats["std"]}')
print(f'ImageNet mean: {norm_stats["imagenet_mean"]}')
print(f'ImageNet std : {norm_stats["imagenet_std"]}')
print(f'Saved to: {norm_path}')

In [ ]:
# ── 9. Report disk usage ──────────────────────────────────────────────────
import shutil

total_bytes, used_bytes, free_bytes = shutil.disk_usage('/kaggle/working')
n_npys = len(list(OUT_DIR.glob('*.npy')))
cache_bytes = sum(f.stat().st_size for f in OUT_DIR.glob('*.npy'))

print('─' * 40)
print(f'NPY files created : {n_npys:,}')
print(f'Cache size        : {cache_bytes/1e9:.2f} GB')
print(f'Working dir used  : {used_bytes/1e9:.2f} GB')
print(f'Working dir free  : {free_bytes/1e9:.2f} GB')
print('─' * 40)
print()
if SUBSET_FRAC < 1.0:
    print('⚠️  PILOT MODE — you processed a subset.')
    print('    If health check passes, set SUBSET_FRAC = 1.0 and re-run.')
    print('    Already-cached files will be skipped (resume-safe).')
else:
    print('NEXT STEPS:')
    print('1. Click "Save Version" → "Save & Run All" to commit this output')
    print('2. In Notebook 03, click "Add Data" → "Notebook Output Files"')
    print('   and search for this notebook to add the NPY cache as input')

In [ ]:
# ── 10. HEALTH CHECK — automated output validation ───────────────────────
import json as _json_hc

errors = []

# Check manifest
manifest_hc = Path('/kaggle/working/manifest.csv')
if not manifest_hc.exists():
    errors.append('manifest.csv is MISSING')
else:
    mf = pd.read_csv(manifest_hc)
    required_cols = ['image_id', 'patient_id', 'split', 'any', 'npy_path']
    for col in required_cols:
        if col not in mf.columns:
            errors.append(f'manifest.csv missing column: {col}')
    if 'patient_id' in mf.columns and 'split' in mf.columns:
        train_p = set(mf[mf['split'] == 'train']['patient_id'])
        val_p   = set(mf[mf['split'] == 'val']['patient_id'])
        if len(train_p & val_p) > 0:
            errors.append(f'PATIENT LEAKAGE: {len(train_p & val_p)} patients in both splits')
    # Adjust threshold for pilot mode
    min_rows = 1000 if SUBSET_FRAC < 1.0 else 100000
    if len(mf) < min_rows:
        errors.append(f'manifest.csv has only {len(mf)} rows — expected {min_rows}+')

# Check normalization stats
norm_hc = Path('/kaggle/working/normalization_stats.json')
if not norm_hc.exists():
    errors.append('normalization_stats.json is MISSING')
else:
    ns = _json_hc.load(open(norm_hc))
    if 'mean' not in ns or 'std' not in ns:
        errors.append('normalization_stats.json missing mean/std')

# Check NPY cache
npy_count = len(list(OUT_DIR.glob('*.npy')))
if npy_count == 0:
    errors.append('No NPY files found in cache directory')

# Spot-check one file for shape/dtype/range
spot = list(OUT_DIR.glob('*.npy'))[:1]
if spot:
    arr = np.load(str(spot[0]))
    if arr.shape != (IMG_SIZE, IMG_SIZE, 3):
        errors.append(f'NPY shape mismatch: {arr.shape} != ({IMG_SIZE},{IMG_SIZE},3)')
    if arr.dtype != np.float32:
        errors.append(f'NPY dtype mismatch: {arr.dtype} != float32')
    if arr.min() < 0 or arr.max() > 1:
        errors.append(f'NPY value range invalid: [{arr.min():.4f}, {arr.max():.4f}]')

# Save health check result
health = {
    'notebook'        : '02_preprocess_cache',
    'status'          : 'PASS' if not errors else 'FAIL',
    'errors'          : errors,
    'mode'            : 'pilot' if SUBSET_FRAC < 1.0 else 'full',
    'manifest_rows'   : len(mf) if manifest_hc.exists() else 0,
    'npy_count'       : npy_count,
    'patient_leakage' : False,
}

with open('/kaggle/working/health_check_nb02.json', 'w') as f:
    _json_hc.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
    raise RuntimeError(f'NB02 health check failed with {len(errors)} error(s)')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Mode       : {"PILOT" if SUBSET_FRAC < 1.0 else "FULL"}')
    print(f'   Manifest   : {health["manifest_rows"]:,} rows')
    print(f'   NPY cache  : {health["npy_count"]:,} files')
    print(f'   Patient leak: None')
    print(f'   Norm stats  : saved')